In [2]:
# =============================================================================
# NOTEBOOK: 08_governance_report.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — Governance Layer: consolidating operational-control artifacts
#
# PURPOSE: Make the governance contribution CONCRETE. The five governance
# functions in the governance figure are not aspirational — each is backed by an
# artifact the pipeline already produces. This notebook gathers those scattered
# assets into one governance report so the deployed ROI application is auditable,
# reproducible, costed, reliability-monitored, assumption-versioned, and gated by
# human accountability.
#
# Governance function  ->  backing artifact
#   1 Audit Trail            Agent-1 rationales + Agent-2 source/[MISSING] tags
#   2 Reproducibility        run_manifest (seed, temperature policy, models)
#   3 Cost & Usage           cost_ledger.json (per-stage spend, model tiers)
#   4 Reliability Monitoring Agent-2 Self-Consistency CV (threshold violations)
#   5 Assumption Registry    cost model + priors + partial_retain (versioned)
#   + Human-in-the-Loop Gate Agent-2 clarifying questions (review queue)
#
# Output:
#   artifacts/governance/governance_report.json
#   artifacts/governance/audit_trail.csv
#   artifacts/governance/review_queue.csv
#   artifacts/governance/assumption_registry.json
#
# Every input is loaded defensively; missing sources degrade gracefully so the
# notebook runs at any pipeline stage. All code/labels English for submission.
# =============================================================================


# %%
# =============================================================================
# Cell 1. Paths + defensive artifact loader
# =============================================================================
import json
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
INFER = ARTIFACTS / "inference"
GOV = ARTIFACTS / "governance"
GOV.mkdir(parents=True, exist_ok=True)


def rel(p):
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


def load_opt(path: Path):
    """Load JSON if present, else None (governance must not hard-fail)."""
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"[WARN] could not parse {rel(path)}: {e}")
    return None


manifest = load_opt(ARTIFACTS / "run_manifest.json")
A1  = load_opt(INFER / "agent1_work_units.json")
A2  = load_opt(INFER / "agent2_time.json")
A3  = load_opt(INFER / "agent3_roi.json")
G   = load_opt(INFER / "process_graph.json")
AX5 = load_opt(INFER / "five_axis_report.json")
LEDGER = load_opt(ARTIFACTS / "cost_ledger.json")

present = {name: (obj is not None) for name, obj in [
    ("run_manifest", manifest), ("agent1", A1), ("agent2", A2),
    ("agent3", A3), ("process_graph", G), ("five_axis", AX5),
    ("cost_ledger", LEDGER)]}
print("[INFO] Artifact availability:")
for k, v in present.items():
    print(f"   {'OK ' if v else '-- '} {k}")


# %%
# =============================================================================
# Cell 2. GOVERNANCE 1 — Audit Trail
#
# One row per pipeline decision, carrying the human-auditable justification:
# the automatability grade + its rationale (Agent 1), and the time-estimate
# provenance (Agent 2 source tag + confidence + [MISSING] status).
# =============================================================================
audit_rows = []
if A1:
    est_by_id = {e["id"]: e for e in (A2["estimates"] if A2 else [])}
    for u in A1["work_units"]:
        e = est_by_id.get(u["id"], {})
        audit_rows.append({
            "unit_id": u["id"],
            "task": u["name"],
            "actor": u.get("actor"),
            "system": u.get("system"),
            "grade": u.get("auto_grade"),
            "grade_rationale": u.get("auto_rationale"),
            "time_source": e.get("source"),
            "time_confidence": e.get("confidence"),
            "minutes_per_case": e.get("minutes_per_case"),
            "reliability_cv": e.get("cv_minutes"),
            "missing_fields": ", ".join(
                f for f in ("actor", "system")
                if u.get(f) == (manifest or {}).get("missing_sentinel", "[MISSING]")
            ),
        })
    audit_df = pd.DataFrame(audit_rows)
    audit_path = GOV / "audit_trail.csv"
    audit_df.to_csv(audit_path, index=False, encoding="utf-8-sig")
    print(f"[GOV 1 · Audit Trail] {len(audit_df)} decisions logged -> "
          f"{rel(audit_path)}")
    print(f"   rationale coverage: "
          f"{audit_df['grade_rationale'].astype(bool).mean()*100:.0f}%")
else:
    audit_df = pd.DataFrame()
    print("[GOV 1 · Audit Trail] SKIPPED (agent1 output missing).")


# %%
# =============================================================================
# Cell 3. GOVERNANCE 2 — Reproducibility Control
#
# Snapshot the settings that make a run repeatable: seed, per-agent temperature
# policy, model roster, and a content hash of the interview input.
# =============================================================================
reproducibility = {"available": bool(manifest)}
if manifest:
    interview_rel = manifest.get("paths", {}).get("interview")
    interview_hash = None
    if interview_rel:
        ip = ROOT / interview_rel
        if ip.exists():
            interview_hash = hashlib.sha256(
                ip.read_bytes()).hexdigest()[:16]
    reproducibility.update({
        "seed": manifest.get("seed"),
        "models": manifest.get("models"),
        "default_tier": manifest.get("default_tier"),
        "temperature_policy": {
            k: v.get("temperature")
            for k, v in manifest.get("pipeline_cfg", {}).items()
        },
        "interview_sha256_16": interview_hash,
    })
    print("[GOV 2 · Reproducibility]")
    print(f"   seed={reproducibility['seed']}  "
          f"temp_policy={reproducibility['temperature_policy']}")
    print(f"   interview hash={interview_hash}")
else:
    print("[GOV 2 · Reproducibility] SKIPPED (run_manifest missing).")


# %%
# =============================================================================
# Cell 4. GOVERNANCE 3 — Cost & Usage Monitoring
#
# The ledger records only the spend of the run that first populated it; cached
# re-runs report $0. To report the TRUE analysis cost regardless of cache state,
# we also reconstruct spend directly from the LLM cache files, each of which
# stored its original cost_usd at first call. We report both views and use the
# larger as the authoritative one-time analysis cost.
# =============================================================================
CACHE_DIR = ARTIFACTS / "llm_cache"

# View A: the ledger (may be $0 on cached re-runs).
ledger_total = round(sum(v.get("total_usd", 0.0) for v in LEDGER.values()), 6) \
    if LEDGER else 0.0
ledger_calls = int(sum(v.get("n_calls", 0) for v in LEDGER.values())) if LEDGER else 0

# View B: reconstruct from cache files (each holds its original cost_usd).
cache_total = 0.0
cache_files = 0
tier_breakdown: dict = {}
if CACHE_DIR.exists():
    for cf in CACHE_DIR.glob("*.json"):
        try:
            c = json.loads(cf.read_text(encoding="utf-8"))
        except Exception:
            continue
        cost = c.get("cost_usd") or 0.0
        cache_total += cost
        cache_files += 1
        tier = c.get("tier", "unknown")
        tier_breakdown[tier] = round(tier_breakdown.get(tier, 0.0) + cost, 6)
cache_total = round(cache_total, 6)

# Authoritative one-time analysis cost = the larger, non-zero view.
authoritative = max(ledger_total, cache_total)
cost_source = ("ledger" if ledger_total >= cache_total and ledger_total > 0
               else "cache reconstruction")

cost_monitor = {
    "available": bool(LEDGER) or cache_files > 0,
    "ledger_total_usd": ledger_total,
    "ledger_calls": ledger_calls,
    "cache_reconstructed_total_usd": cache_total,
    "cache_files": cache_files,
    "tier_breakdown_usd": tier_breakdown,
    "authoritative_one_time_cost_usd": authoritative,
    "cost_source": cost_source,
    "per_stage": LEDGER or {},
}

print("[GOV 3 · Cost & Usage]")
if LEDGER:
    for stage, v in LEDGER.items():
        print(f"   ledger {stage:14s}: ${v.get('total_usd',0):.5f} "
              f"({v.get('n_calls',0)} calls)")
print(f"   ledger total          : ${ledger_total:.5f} ({ledger_calls} calls)")
print(f"   cache-reconstructed   : ${cache_total:.5f} "
      f"({cache_files} cached calls)  tiers={tier_breakdown}")
print(f"   authoritative cost    : ${authoritative:.5f}  (source: {cost_source})")


# %%
# =============================================================================
# Cell 5. GOVERNANCE 4 — Reliability Monitoring (CV threshold violations)
#
# Flag estimates whose Self-Consistency dispersion exceeds a governance
# threshold; these are the nodes an operator should watch or re-sample.
# =============================================================================
CV_THRESHOLD = 0.35   # governance policy parameter
reliability_mon = {"available": bool(A2), "cv_threshold": CV_THRESHOLD}
if A2:
    viol = [{"unit_id": e["id"], "task": e["name"],
             "cv": e.get("cv_minutes"), "confidence": e.get("confidence")}
            for e in A2["estimates"]
            if (e.get("cv_minutes") or 0) > CV_THRESHOLD]
    reliability_mon.update({
        "n_nodes": len(A2["estimates"]),
        "n_violations": len(viol),
        "violation_rate_pct": round(len(viol) / max(len(A2["estimates"]), 1) * 100, 1),
        "violations": viol,
    })
    print("[GOV 4 · Reliability Monitoring]")
    print(f"   {len(viol)}/{len(A2['estimates'])} nodes exceed CV>"
          f"{CV_THRESHOLD} ({reliability_mon['violation_rate_pct']}%)")
    for v in viol[:5]:
        print(f"   - {v['task'][:40]:40s} CV={v['cv']}")
else:
    print("[GOV 4 · Reliability Monitoring] SKIPPED (agent2 output missing).")


# %%
# =============================================================================
# Cell 6. GOVERNANCE 5 — Assumption Registry (versioned, hashed)
#
# All editable economic/behavioral assumptions in one registry with a content
# hash, so any change to priors, cost model, or partial-retain is traceable.
# =============================================================================
assumptions = {}
if manifest:
    assumptions["cost_model"] = manifest.get("cost_model")
if A2:
    assumptions["shape_prior_min"] = A2["params"].get("shape_prior_min")
    assumptions["default_cases_per_month"] = A2["params"].get("default_cases_per_month")
    assumptions["qualitative_weight"] = A2["params"].get("qualitative_weight")
if G:
    assumptions["partial_retain"] = G["params"].get("partial_retain")

reg_hash = hashlib.sha256(
    json.dumps(assumptions, sort_keys=True, ensure_ascii=False).encode()
).hexdigest()[:16]
assumption_registry = {
    "version_hash": reg_hash,
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "assumptions": assumptions,
}
reg_path = GOV / "assumption_registry.json"
reg_path.write_text(json.dumps(assumption_registry, ensure_ascii=False, indent=2),
                    encoding="utf-8")
print("[GOV 5 · Assumption Registry]")
print(f"   version hash={reg_hash}  keys={list(assumptions)}")
print(f"   -> {rel(reg_path)}")


# %%
# =============================================================================
# Cell 7. HUMAN-IN-THE-LOOP GATE — review queue
#
# The clarifying questions Agent 2 raised become an explicit review queue: items
# an expert must answer/sign off before the ROI is released. This operationalizes
# the accountability gate at the bottom of the governance figure.
# =============================================================================
queue_rows = []
if A2:
    for q in A2.get("clarifying_questions", []):
        queue_rows.append({
            "unit_id": q.get("id"),
            "task": q.get("name"),
            "reason": q.get("why"),
            "question_for_expert": q.get("question"),
            "status": "PENDING_REVIEW",
        })
review_df = pd.DataFrame(queue_rows)
if not review_df.empty:
    rq_path = GOV / "review_queue.csv"
    review_df.to_csv(rq_path, index=False, encoding="utf-8-sig")
    print(f"[GATE · Human-in-the-Loop] {len(review_df)} item(s) pending expert "
          f"sign-off -> {rel(rq_path)}")
else:
    print("[GATE · Human-in-the-Loop] no pending items (or agent2 missing).")


# %%
# =============================================================================
# Cell 8. Consolidate into a single governance report
# =============================================================================
governance_report = {
    "artifact": "governance_report",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "artifact_availability": present,
    "functions": {
        "1_audit_trail": {
            "n_decisions": len(audit_df),
            "rationale_coverage_pct": round(
                audit_df["grade_rationale"].astype(bool).mean() * 100, 1)
            if not audit_df.empty else None,
            "file": "governance/audit_trail.csv" if not audit_df.empty else None,
        },
        "2_reproducibility": reproducibility,
        "3_cost_usage": {
            "authoritative_one_time_cost_usd":
                cost_monitor.get("authoritative_one_time_cost_usd"),
            "cost_source": cost_monitor.get("cost_source"),
            "tier_breakdown_usd": cost_monitor.get("tier_breakdown_usd"),
            "cache_files": cost_monitor.get("cache_files"),
        },
        "4_reliability_monitoring": reliability_mon,
        "5_assumption_registry": {
            "version_hash": reg_hash,
            "file": "governance/assumption_registry.json",
        },
        "human_in_the_loop_gate": {
            "n_pending": len(review_df),
            "file": "governance/review_queue.csv" if not review_df.empty else None,
        },
    },
}
rep_path = GOV / "governance_report.json"
rep_path.write_text(json.dumps(governance_report, ensure_ascii=False, indent=2),
                    encoding="utf-8")
print(f"\n[INFO] Governance report -> {rel(rep_path)}")

# One-line governance posture summary.
funcs = governance_report["functions"]
active = sum(1 for k, v in funcs.items()
             if isinstance(v, dict) and v.get("available", True)
             and v not in ({}, None))
print(f"[INFO] Governance posture: {len(funcs)} functions consolidated; "
      f"the ROI application is auditable, reproducible, cost-monitored, "
      f"reliability-monitored, assumption-versioned, and human-gated.")

[INFO] Artifact availability:
   OK  run_manifest
   OK  agent1
   OK  agent2
   OK  agent3
   OK  process_graph
   OK  five_axis
   OK  cost_ledger
[GOV 1 · Audit Trail] 46 decisions logged -> artifacts\governance\audit_trail.csv
   rationale coverage: 100%
[GOV 2 · Reproducibility]
   seed=42  temp_policy={'agent1_extract': 0.0, 'agent2_time': 0.7, 'agent3_roi': 0.0}
   interview hash=18b4a57b444b4cbf
[GOV 3 · Cost & Usage]
   ledger 01_agent1     : $0.00000 (0 calls)
   ledger total          : $0.00000 (0 calls)
   cache-reconstructed   : $0.01651 (12 cached calls)  tiers={'weak': 0.016509, 'unknown': 0.0}
   authoritative cost    : $0.01651  (source: cache reconstruction)
[GOV 4 · Reliability Monitoring]
   10/46 nodes exceed CV>0.35 (21.7%)
   - File evaluation request                  CV=0.387
   - Check applicant company's basic informat CV=0.396
   - Fill in technology summary sheet         CV=0.416
   - Assign on-site or phone inspector        CV=0.399
   - Check previous tech